In [0]:
%run ./src/foodquest_pnl

In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="fact_financial_pnl",
    choices=["discount", "sales", "fact_financial_pnl", "fact_financial_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.pnl_actual_flat_data limit 1

In [0]:

df = spark.read.table('fq_dev_catalog.bronze.pnl_actual_flat_data')

from pyspark.sql.functions import *
df_filtered = df.filter(((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2025)) 
                        # | ((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2026)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2025)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2026))
)
df_filtered.display()



In [0]:
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df(df):

    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")
    df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


    # Step 1: Joined both master tables upfront
    df_all_masters = df.join(
        df_coa_master_distinct, 
        df_coa_master_distinct["mapped_name"].cast("string") == df["Column 1"], 
        'inner'
    ).join(
        df_location_master,
        col("Store name") == df_location_master["excel_p&l_name"],
        'left'
    )

    df_all_masters = df_all_masters.withColumnsRenamed(
        {
            "File name": "month",
            "Year": "year",
            # "Column 1": "account_name",
            "Act": "amount",
            "Store name": "location"
        }
    )

    

    # Step 2: Created detail rows with all columns
    df_detail = df_all_masters.select(
        col("location"),
        col("month"),
        col("year"),
        col("account_name"),
        col("name"),
        col("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("amount"),
        # Location master columns
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Detail").alias("sum_order")
    )

    # Step 3: Created aggregations with location master columns in groupBy
    df_subgroup_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group", "sub_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("sub_group")).alias("account_name"),
        concat(lit("Total "), col("sub_group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("amount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Sub Total").alias("sum_order")
    )

    df_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("group")).alias("account_name"),
        concat(lit("Total "), col("group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        col("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Total").alias("sum_order")
    )

    df_majour_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("majour_group")).alias("account_name"),
        concat(lit("Total "), col("majour_group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        lit("N/A").alias("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Grand Total").alias("sum_order")
    )

    # Gross Profit = Total Sales - Total Purchases (sum_order 4)
    df_gross_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        (sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        lit("Gross Profit").alias("account_name"),
        lit("Gross Profit").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Gross Profit").alias("majour_group"),
        lit("N/A").alias("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Grand Total").alias("sum_order")
    )
    
    # Operating Profit = Gross Profit - Total Overheads (sum_order 4)
    df_operating_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        ((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("Operating Profit").alias("account_name"),
        lit("Operating Profit").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Operating Profit").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("city"),
        lit("Grand Total").alias("sum_order")
    )

    # EBITDA = Operating Profit + Total Depreciation & Amortization 
    df_ebitda = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) +
        sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("EBITDA").alias("account_name"),
        lit("EBITDA").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("EBITDA").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("city"),
        lit("Grand Total").alias("sum_order")
    )

    # Net Profit = Operating Profit - Total Finance Costs - Total Tax (sum_order 4)
    df_net_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "store_type", "city"
    ).agg(
        (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0)) -
        sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("Net Profit / (Loss)").alias("account_name"),
        lit("Net Profit / (Loss)").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Net Profit / (Loss)").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("city"),
        lit("Grand Total").alias("sum_order")
    )

    # Step 4: Update union to include all calculated rows
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))

    # Step 5: Extracted year and prepare for PY calculations
    df_current = df_combined.withColumn("year", col("year"))

    # Step 6: Calculated Previous Year Sales (PY) - Self join
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("location").alias("py_location"),
        col("account_name").alias("py_account_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )

    df_with_py = df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.location") == col("py_location")) &  
        (col("curr.account_name") == col("py_account_name")) &  
        (col("curr.month") == col("py_month")),  
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )

    # Step 7: Calculated Net Sales with window functions
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")

    df_with_calculations = df_with_py.withColumn(
        "Actual Net Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "PY Netsales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "Location Actual Net Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "Location PY Net Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "Brand Act Nets Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "Brand PY NetSales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "Company Act Nets Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_company)
    ).withColumn(
        "Company PY NetSales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_company)
    )

    # Step 8: Created final output with all required columns
    df_final = df_with_calculations.select(
        col("account_name"),
        col("name"),
        col("majour_group").alias("Major Group"),
        col("group").alias("Group"),
        col("sub_group").alias("sub_group"),
        lit(None).cast("string").alias("Alternate Group"),
        col("accoun_type").alias("accounttype"),
        col("netsuite_location_name").alias("Location"),
        col("type").alias("Location type (HO/Store)"),
        col("location_id").alias("Location Code"),
        col("brand_id").alias("Brand"),
        col("company_id").alias("Company"),
        lit(None).cast("string").alias("Cluster"),
        col("store_type").alias("Location Mode (Mall/Drive thru/Stand alone)"),
        col("city").alias("Emirates"),
        col("sum_order"),
        col("amount").alias("Actual Value"),
        col("year").alias("Year"),
        col("month").alias("Month"),
        col("Actual Net Sales"),
        col("PY Netsales"),
        col("Brand Act Nets Sales"),
        col("Brand PY NetSales"),
        col("Company Act Nets Sales"),
        col("Company PY NetSales"),
    )
    fact_financial_pnl = to_snake_case_df(df_final)
    return fact_financial_pnl

In [0]:
df_final = enrich_df(df_filtered)
df_final.display()

In [0]:
from pyspark.sql.functions import *
import re
import sys
sys.path.append("/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/src")

from foodquest_pnl import (
    create_total_row, 
    create_calculated_metric, 
    add_previous_year_data,
    add_net_sales_calculations,
    get_dimension_columns
)

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df(df):

    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")
    df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


    # Step 1: Joined both master tables upfront
    df_all_masters = df.join(
        df_coa_master_distinct, 
        df_coa_master_distinct["mapped_name"].cast("string") == df["Column 1"], 
        'inner'
    ).join(
        df_location_master,
        col("Store name") == df_location_master["excel_p&l_name"],
        'left'
    )

    df_all_masters = df_all_masters.withColumnsRenamed(
        {
            "File name": "month",
            "Year": "year",
            # "Column 1": "account_name",
            "Act": "amount",
            "Store name": "location"
        }
    )
    dimension_cols = get_dimension_columns()
    

    # Step 2: Create detail rows
    df_detail = df_all_masters.select(
        col("location"), col("month"), col("year"),
        col("account_name"), col("name"), col("accoun_type"),
        col("majour_group"), col("group"), col("sub_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit("Detail").alias("Detail/Total")
    )
    
    # Step 3: Create total rows at different aggregation levels
    df_subgroup_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group", "group", "sub_group"] + dimension_cols,
        "sub_group",
        "Total",
        dimension_cols
    )
    
    df_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group", "group"] + dimension_cols,
        "group",
        "Total",
        dimension_cols
    )
    
    df_majour_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group"] + dimension_cols,
        "majour_group",
        "Grand Total",
        dimension_cols
    )
    
    # Step 4: Create calculated metrics
    df_gross_profit = create_calculated_metric(
        df_all_masters,
        "Gross Profit",
        abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_operating_profit = create_calculated_metric(
        df_all_masters,
        "Operating Profit",
        abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_ebitda = create_calculated_metric(
        df_all_masters,
        "EBITDA",
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) +
        abs(sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_net_profit = create_calculated_metric(
        df_all_masters,
        "Net Profit/(Loss)",
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) -
        abs(sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0))) -
        abs(sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    # Step 5: Union all dataframes
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))
    
    # Step 6: Add previous year data
    df_with_py = add_previous_year_data(df_combined)
    
    # Step 7: Calculate net sales metrics
    df_with_calculations = add_net_sales_calculations(df_with_py)
    
    # Step 8: Select final columns and convert to snake case
    df_final = df_with_calculations.select(
        col("account_name"), col("name"), col("accoun_type"),
        col("majour_group"), col("group"), col("sub_group"),
        *[col(c) for c in dimension_cols],
        col("Detail/Total"), col("amount"), col("year"), col("month"),
        col("actual_net_sales"), col("py_net_sales"),
        col("brand_act_net_sales"), col("brand_py_net_sales"),
        col("company_act_net_sales"), col("company_py_net_sales")
    )
    
    df_final = to_snake_case_df(df_final)
    
    # Step 9: Join with sort order and final selection
    df_sort = df_coa_master.select("account_name", "sort_order", "calculation_type")
    df_final_sort = df_final.join(
        df_sort, 
        df_sort["account_name"].cast("string") == df_final.account_name,
        'inner'
    ).drop(df_sort["account_name"]).select(
        col("account_name"), col("name"),
        col("majour_group"), col("group"), col("sub_group"), col("accoun_type"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"),
        col("parent_company"), col("country_code"), col('zone'), col("city"),
        col("amount"), col("year"), col("month"),
        col("actual_net_sales"), col("py_net_sales"),
        col("brand_act_net_sales"), col("brand_py_net_sales"),
        col("company_act_net_sales"), col("company_py_net_sales"),
        col("detail/total"), col('sort_order'), col('calculation_type')
    )
    
    return df_final_sort.orderBy(
        "parent_company", "company_id", "brand_id", 
        "netsuite_location_name", 'year', 'month', 'sort_order'
    )

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_catalog.silver.fact_financial_pnl (
  account_name STRING,
  account_number STRING,
  major_group STRING,
  group_name STRING,
  sub_group STRING,
  alternate_group STRING,
  account_type STRING,
  location STRING,
  location_type STRING COMMENT 'HO/Store',
  location_code STRING,
  brand STRING,
  company STRING,
  cluster STRING,
  location_mode STRING COMMENT 'Mall/Drive thru/Stand alone',
  emirates STRING,
  detail_total_grand_total STRING,
  sort_order INT,
  actual_value DECIMAL(18,2),
  budget DECIMAL(18,2),
  previous_year_sales DECIMAL(18,2),
  year INT,
  month INT,
  forecast DECIMAL(18,2),
  calculation_type STRING,
  actual_net_sales DECIMAL(18,2),
  budget_net_sales DECIMAL(18,2),
  py_net_sales DECIMAL(18,2),
  brand_act_net_sales DECIMAL(18,2),
  brand_budget_net_sales DECIMAL(18,2),
  brand_py_net_sales DECIMAL(18,2),
  company_act_net_sales DECIMAL(18,2),
  company_budget_net_sales DECIMAL(18,2),
  company_py_net_sales DECIMAL(18,2)
)
USING DELTA
CLUSTER BY (year, month, company, brand)
LOCATION 'abfss://fq-dev-silver-container@fqadfstoragedev.dfs.core.windows.net/external/fact_financial_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
def merge_stream_fact_financial_pnl(df, i):
    try:
        exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
        
        fact_financial_pnl_upsert = enrich_json(exploded_df)
        fact_financial_pnl_upsert.createOrReplaceTempView("fact_financial_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
            USING (
                SELECT *
                FROM fact_financial_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.location = source.location
                AND target.account_name = source.account_name
                AND target.sort_order = source.sort_order
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print(f"Successfully merged batch {i}")

    # df.sparkSession.sql("""
    #     MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
    #     USING (
    #         SELECT *
    #         FROM (
    #             SELECT *, 
    #                 ROW_NUMBER() OVER (
    #                     PARTITION BY year, month, location, account_name, sort_order
    #                     ORDER BY year DESC  -- or add a load_time column
    #                 ) as rank
    #             FROM fact_financial_pnl_upsert_microbatch
    #         )
    #         WHERE rank = 1
    #     ) as source
    #     ON target.year = source.year
    #         AND target.month = source.month
    #         AND target.location = source.location
    #         AND target.account_name = source.account_name
    #         AND target.sort_order = source.sort_order
    #     WHEN MATCHED THEN UPDATE SET *
    #     WHEN NOT MATCHED THEN INSERT *
    # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_fact_financial_pnl/schema_fact_financial_pnl')
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_fact_financial_pnl)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_fact_financial_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.fact_financial_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.fact_financial_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()